In [ ]:
# @@ENV@@ — Keras backend, visible GPU and global seed
import os

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)


import kagglesdk.kaggle_env as kaggle_env

if not hasattr(kaggle_env, "get_web_endpoint"):
    kaggle_env.get_web_endpoint = kaggle_env.get_endpoint

import kagglehub
import keras
import keras_cv

print("GPUs:", gpus)

2026-08-07 15:40:00.913551: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-07 15:40:00.924371: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786110000.936792 2837357 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786110000.940664 2837357 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-07 15:40:00.954155: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### **Setup**

In [ ]:
# @@CFG@@ — the knobs the post-hoc needs; split and sampling live in the frozen frames
class CFG:
    seed = 49
    image_size = (256, 256)
    batch_size = 64
    num_classes = 1
    xai_balance = "equal"   # half per class: the sample of the study
    xai_n = None            # None = the largest this split allows (140 here); a number is exploratory

keras.utils.set_random_seed(CFG.seed)

In [ ]:
# @@LIBS@@ - Common libraries used across the post-hoc pipeline.
import glob, gc, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

In [ ]:
# @@PATHS@@ — dataset, variant and row identity are declared HERE; every path derives from them
import sys

BASE_PATH = os.environ.get("DUM_BASE", os.environ.get("ISIC24_BASE", "/mnt/disk1/home/cgiovannoni/mod-attendance"))
sys.path.insert(0, os.path.join(BASE_PATH, "src")) if os.path.join(BASE_PATH, "src") not in sys.path else None

from isic24 import paths as pth
from isic24 import xai_subset as xs

pth.DATASET, pth.VARIANT = "isic24", "enhanced"   # BRSET/Symile: change these two, nothing else
xs.ID_COL = "isic_id"                             # row identity column of this dataset

DATA_DIR = str(pth.data_dir())                    # images (hdf5) live at dataset level
DATA_PATH = str(pth.data_path())                  # metadata of this variant
ART_PATH = str(pth.art_path())
FEATURE_SPACE_PATH = os.path.join(ART_PATH, "feature_space")
RUNS_PATH = os.path.join(ART_PATH, "runs")
SUMMARIES_PATH = os.path.join(ART_PATH, "summaries")

TRAIN_META = os.path.join(DATA_PATH, "enhanced_train_metadata_clean.parquet")
TEST_META = os.path.join(DATA_PATH, "enhanced_test_metadata_clean.parquet")
TRAIN_HDF5 = os.path.join(DATA_DIR, "train-image.hdf5")
TEST_HDF5 = os.path.join(DATA_DIR, "test-image.hdf5")

print(f"dataset={pth.DATASET}/{pth.VARIANT} | id={xs.ID_COL}\n  data {DATA_PATH}\n  art  {ART_PATH}")

dataset=isic24/enhanced | id=isic_id
  data /mnt/disk1/home/cgiovannoni/mod-attendance/data/isic24/enhanced
  art  /mnt/disk1/home/cgiovannoni/mod-attendance/artifacts/isic24/enhanced


In [ ]:
# @@FRAMES@@ — load the frames the training froze; this notebook never rebuilds the split
import json as _json
import pandas as pd

validation_df = pd.read_parquet(os.path.join(ART_PATH, "evaluation_frame.parquet"))
training_df = pd.read_parquet(os.path.join(ART_PATH, "training_frame.parquet"))

_snap = _json.load(open(os.path.join(ART_PATH, "validation_snapshot.json")))
_live = [str(v) for v in xs.row_ids(validation_df)]
_frozen = [str(v) for v in _snap["row_ids"]]
if _live != _frozen:
    _lost, _new = set(_frozen) - set(_live), set(_live) - set(_frozen)
    raise RuntimeError(
        f"[FRAMES] the evaluation frame ({len(_live)} rows) is not the frozen validation "
        f"({len(_frozen)} rows): "
        + ("the same rows in a DIFFERENT ORDER." if not (_lost or _new)
           else f"{len(_lost)} missing, {len(_new)} new.")
        + " Every val_index in the XAI artifacts is a position into the frozen frame.")

print(f"[FRAMES] evaluation {validation_df.shape}, {int(validation_df.target.sum())} positive, "
      f"prevalence {validation_df.target.mean():.4f} | training {training_df.shape}")

In [ ]:
# @@CUSTOM_LAYERS@@ — register the shared Keras layers, so load_model needs no custom_objects
import sys, os
for _p in (os.path.join(BASE_PATH, "src"),
           os.path.join(BASE_PATH, "notebooks", "modality_attendance")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from isic24.mod_attendance import (
    ConstantScale, ScalarMultiply, LearnableAlpha, FusionWithGamma, GateEntropyLoss,
)
from feature_space_guard import assert_fs_matches_model   # PH-13: this file belongs in the package

[FS GUARD] functions defined. `feature_space` not in scope yet — call assert_feature_space_canonical(feature_space) after cell 16, and assert_fs_matches_model(model, feature_space) after each load_model.


In [6]:
# @@PARAMETERS@@ — canonical study dirs, budgets, evaluation set, XAI sample sizes
# Read by every cell below; loads no model and touches no GPU.
CANONICAL = {
    "vanilla":             os.path.join(RUNS_PATH, "vanilla", "20260604-210304"),
    "fixed_scaling":       os.path.join(RUNS_PATH, "fixed_scaling", "20260622-112038"),
    "learnable_scaling":   os.path.join(RUNS_PATH, "learnable_scaling", "20260619-184157"),
    "loss_regularization": os.path.join(RUNS_PATH, "loss_regularization", "20260610-095420"),
    "controlled_fusion":   os.path.join(RUNS_PATH, "controlled_fusion", "20260616-102253"),
    "adaptive_gating":     os.path.join(RUNS_PATH, "adaptive_gating", "20260629-185216"),
}
cf_study_dir = CANONICAL["controlled_fusion"]   # gamma sweep writes here
ag_study_dir = CANONICAL["adaptive_gating"]     # gate diagnostics write here

FAITH_BUDGETS = (0.05, 0.10, 0.20, 0.40, 0.60)   # q points the model is re-scored at (GPU)
Q_MAX_REPORT = [0.20, 0.60]                      # integration cutoffs, re-read from saved curves

EVAL_SET   = "canonical"     # "canonical" = the fold-0 set the runs were evaluated on | "custom"
EVAL_FOLDS = [0, 1]          # used only when EVAL_SET == "custom"; folds 1-4 are training data
FAITH_N_SAMPLES = 128        # hard ceiling of the saved tabular/joint SHAP arrays
FAITH_N_SAMPLES_TARGET_NEW_DATASETS = 256
FAITH_N_SAMPLES_EXPLORE = None

# Which runs the faithfulness loop processes (see the faithfulness cell for how they are used).
FAITH_ONLY_IMPL = None    # e.g. ["vanilla", "adaptive_gating"]; None = all discovered
FAITH_MAX_RUNS  = None    # e.g. 5 for a smoke test
FAITH_OVERWRITE = False   # False = reuse inline-computed faithfulness.json when present
FAITH_DRY_RUN   = False   # True = list what would run, load no models

# Small CSV summaries (no models loaded here).
cf_combined = pd.read_csv(os.path.join(cf_study_dir, "combined_summary.csv"))
ag_combined = pd.read_csv(os.path.join(ag_study_dir, "combined_summary.csv"))
print("params set | cf_combined", cf_combined.shape, "| ag_combined", ag_combined.shape)

# Run selection for the LORE regeneration section below; ~21 min per run.
LORE_DRY_RUN    = False
LORE_ONLY_IMPL  = ["fixed_scaling", "learnable_scaling", "loss_regularization"]   # None = all
LORE_PER_FAMILY = 1
LORE_MAX_RUNS   = None   # head(n) applied AFTER the filters; None = every selected run
LORE_ONLY_TAG   = ["a0p500", "init_a0p500", "lam1e-03"]   # None = every setting_tag

LORE_N_SAMPLES_EXPLORE = 128   # exploration tier only; the default stays 32 (28 s/sample)


params set | cf_combined (5, 21) | ag_combined (5, 50)


In [ ]:
# @@HDF5@@ — the image store the evaluation frame reads from
import h5py

training_validation_hdf5 = h5py.File(TRAIN_HDF5, 'r')

In [ ]:
# @@FEATURE_SPACE@@ — the space the checkpoints were trained with; the feature spec comes from it
FEATURE_SPACE_DIR = os.path.join(FEATURE_SPACE_PATH, "20260810-192622")

feature_space = keras.saving.load_model(os.path.join(FEATURE_SPACE_DIR, "feature_space.keras"))
assert isinstance(feature_space, keras.utils.FeatureSpace), (
    f"{FEATURE_SPACE_DIR} deserializes to {type(feature_space).__name__}, not a FeatureSpace")

feature_order = list(feature_space.features)   # declared order; SHAP reads attributions by it
FEAT_COLS = feature_order
ENCODED_DIM = int(feature_space.get_encoded_features().shape[-1])

_missing = [c for c in FEAT_COLS if c not in validation_df.columns]
assert not _missing, f"the frozen frames do not carry {len(_missing)} model features: {_missing}"
print(f"FeatureSpace {os.path.basename(FEATURE_SPACE_DIR)} | {len(FEAT_COLS)} features "
      f"-> {ENCODED_DIM} encoded dims")

In [ ]:
# @@DATASET_BUILDERS@@ — decoder and dataset builder; the post-hoc never shuffles nor augments
from tqdm import tqdm

def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        image = tf.io.decode_jpeg(inp["images"])
        image = tf.image.resize(image, size=target_size, method="area")
        image = tf.cast(image, tf.float32) / 255.0
        image = tf.reshape(image, [*target_size, 3])
        return {"images": image, "features": inp["features"]}

    def decode_with_labels(inp, label):
        label = tf.reshape(tf.cast(label, tf.float32), [CFG.num_classes])
        return decode_image(inp), label

    return decode_with_labels if with_labels else decode_image


def build_dataset(row_ids, hdf5, features, labels=None, batch_size=None, cache=True):
    images = [hdf5[row_id][()] for row_id in tqdm(row_ids, desc="Loading images")]
    inputs = {"images": images, "features": features}
    slices = (inputs, labels) if labels is not None else inputs

    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.map(build_decoder(labels is not None), num_parallel_calls=tf.data.AUTOTUNE,
                deterministic=True)
    if cache:
        ds = ds.cache()
    if batch_size is not None:
        ds = ds.batch(batch_size)
    return ds


def _to_feature_dict(df):
    feat_dict = {}
    for col in df.columns:
        series = df[col]
        feat_dict[col] = (series.to_numpy() if pd.api.types.is_numeric_dtype(series.dtype)
                          else series.astype(str).to_numpy())
    return feat_dict

In [ ]:
# @@BUILD_DATASETS@@ — the evaluation set only, decoded once and encoded in frame order
validation_ds = build_dataset(
    xs.row_ids(validation_df), training_validation_hdf5,
    _to_feature_dict(validation_df[FEAT_COLS]), validation_df.target.to_numpy(),
    batch_size=CFG.batch_size,
).map(lambda x, y: ({"images": x["images"], "features": feature_space(x["features"])}, y),
      num_parallel_calls=tf.data.AUTOTUNE, deterministic=True).prefetch(tf.data.AUTOTUNE)

_n = sum(int(x["images"].shape[0]) for x, _ in validation_ds)
assert _n == len(validation_df), f"the dataset yields {_n} rows, the frame holds {len(validation_df)}"
print(f"[EVAL] {_n} rows, {len(FEAT_COLS)} features -> {ENCODED_DIM} encoded dims")

In [ ]:
# @@ENCODED_NAMES@@ — names of the encoded columns in FeatureSpace order
from keras.layers import StringLookup, Discretization, Normalization

encoded_feature_names = []
feature_dim_sizes = {}

for name in feature_order:
    preproc = feature_space.preprocessors[name]

    # 1) Categorical features: StringLookup → one-hot of size vocabulary_size
    if isinstance(preproc, StringLookup):
        vocab = preproc.get_vocabulary()
        # each token gets its own encoded dim
        for token in vocab:
            encoded_feature_names.append(f"{name}__{token}")
        d = len(vocab)

    # 2) Discretized float (age_approx): Discretization → one-hot over bins
    elif isinstance(preproc, Discretization):
        boundaries = preproc.get_config().get("bin_boundaries", [])
        n_bins = len(boundaries) + 1
        for b in range(n_bins):
            encoded_feature_names.append(f"{name}_bin_{b}")
        d = n_bins

    # 3) Normalized numeric or raw float: 1 dim
    elif isinstance(preproc, Normalization):
        encoded_feature_names.append(name)
        d = 1

    else:
        # "float" (already normalized) or any other scalar
        encoded_feature_names.append(name)
        d = 1

    feature_dim_sizes[name] = d

# sanity check
print("per-feature dim sizes:", feature_dim_sizes)
print("len(encoded_feature_names):", len(encoded_feature_names))
print("feature_space encoded dim:", feature_space.get_encoded_features().shape[-1])

assert len(encoded_feature_names) == feature_space.get_encoded_features().shape[-1]

In [ ]:
# @@SETUP_XAI@@ — bind the session context, then adopt the XAI sample once for every engine
from isic24 import xai_faithfulness as xf
from isic24.xai_faithfulness import (
    explanation_faithfulness, stratified_table, pool_by_family, discover_curve_files,
    verify_reintegration, _bootstrap_ci, _load_shap_values,
)
from isic24.xai_driver import run_posthoc_xai
from isic24.xai_engines import _finalize_plot

xf.configure(
    xf.XaiContext(
        feat_cols=FEAT_COLS, training_df=training_df, validation_df=validation_df,
        validation_ds=validation_ds, feature_space=feature_space,
        to_feature_dict=_to_feature_dict, encoded_feature_names=encoded_feature_names,
    ),
    FAITH_N_SAMPLES=FAITH_N_SAMPLES,   # SHAP cost ceiling only; who gets explained is the sample below
    FAITH_BUDGETS=tuple(FAITH_BUDGETS),
    EVAL_IS_CANONICAL=True,            # the frozen evaluation frame is the only set this notebook reads
)

# One central setting per family: the comparison set behind every table and figure.
CENTRAL_SETTINGS = {"vanilla": "baseline", "fixed_scaling": "a0p500",
                    "learnable_scaling": "init_a0p500", "loss_regularization": "lam1e-03",
                    "controlled_fusion": "base", "adaptive_gating": "base"}

# One sample per session, on DISTINCT lesions, shared by SHAP, Grad-CAM, LORE and faithfulness.
xai_subset_idx, xai_subset_info = xs.adopt(
    validation_df, n=CFG.xai_n, balance=CFG.xai_balance, seed=0, snapshot_dir=ART_PATH)
XAI_SUBSET_FP = xai_subset_info["fingerprint"]


In [ ]:
# @@STOP@@ — the setup ends here, everything below explains checkpoints
raise StopIteration("setup complete — the cells below run one at a time")

StopIteration: 

## Feature provenance and site shortcut


In [ ]:
# @@FEATURE_PROVENANCE@@ 1/2 — where the 39 tabular features come from
# Checks FEAT_COLS for diagnostic leakage, splits release from engineered, and measures the
# prevalence spread across sites.

CONTEXT_COLS = ["sex", "age_approx", "anatom_site_general", "tbp_lv_location",
                "combined_anatomical_site", "attribution", "tbp_tile_type"]
MODEL_OUTPUT_COLS = ["tbp_lv_nevi_confidence"]
IMAGE_DERIVED_COLS = [c for c in FEAT_COLS if c not in CONTEXT_COLS]
DIAGNOSTIC_COLS = ["iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5", "lesion_id",
                   "mel_mitotic_index", "mel_thick_mm", "tbp_lv_dnn_lesion_confidence"]

# The enhanced variant adds no measurement: its extra columns are closed-form functions of
# columns the ISIC release already ships, and the raw TBP columns they read are mostly unused.
RELEASE_FEAT_COLS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS
ENGINEERED_FEAT_COLS = NEW_CATEGORICAL_COLUMNS + NEW_NUMERIC_COLUMNS
RAW_TBP_COLS = sorted(c for c in training_df.columns
                      if c.startswith("tbp_lv_") and c not in DIAGNOSTIC_COLS)

leaked = sorted(set(FEAT_COLS) & set(DIAGNOSTIC_COLS))
print(f"[provenance] {len(FEAT_COLS)} features = {len(CONTEXT_COLS)} context + "
      f"{len(IMAGE_DERIVED_COLS)} image-derived, of which model output: {MODEL_OUTPUT_COLS}")
print(f"[leakage]    diagnostic columns in FEAT_COLS: {leaked or 'none'} -> "
      f"{'FAIL' if leaked else 'PASS'}")
print(f"[variant]    {len(RELEASE_FEAT_COLS)} features straight from the ISIC release + "
      f"{len(ENGINEERED_FEAT_COLS)} engineered on top | {len(RAW_TBP_COLS)} raw TBP columns "
      f"exist, {sum(c not in FEAT_COLS for c in RAW_TBP_COLS)} of them never reach the model")

natural_df = pd.read_parquet(TRAIN_META)
resampled_df = pd.concat([training_df, validation_df])
for col in ("attribution", "tbp_tile_type"):
    for label, frame in (("natural", natural_df), ("resampled", resampled_df)):
        p = frame.groupby(col)["target"].mean()
        print(f"[{col:14s} {label:9s}] {len(p)} levels, prevalence "
              f"{p.min():.4f}..{p.max():.4f}, ratio {p.max() / max(p.min(), 1e-9):.1f}x")


In [ ]:
# @@FEATURE_PROVENANCE@@ 2/2 — how much of the tabular AUC is site shortcut, and how much is
# feature engineering rather than information. Gradient boosting on the same split, CPU ~2 min.

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

def _split_encoded(cols):
    both = pd.concat([training_df[cols], validation_df[cols]])
    for c in both.columns:
        if not pd.api.types.is_numeric_dtype(both[c]):
            both[c] = both[c].astype("category").cat.codes
    both = both.astype("float32")
    return both.iloc[:len(training_df)], both.iloc[len(training_df):]

SUBSETS = {
    "attribution alone": ["attribution"],
    "context only": CONTEXT_COLS,
    "image-derived only": IMAGE_DERIVED_COLS,
    "all features": FEAT_COLS,
    "all minus attribution/tile": [c for c in FEAT_COLS
                                   if c not in ("attribution", "tbp_tile_type")],
    "release cols, as we use them": RELEASE_FEAT_COLS,
    "release cols, every raw TBP": sorted(set(RELEASE_FEAT_COLS) | set(RAW_TBP_COLS)),
    "engineered cols only": ENGINEERED_FEAT_COLS,
}
y_tr, y_va = training_df["target"].to_numpy(), validation_df["target"].to_numpy()
for name, cols in SUBSETS.items():
    X_tr, X_va = _split_encoded(cols)
    p = HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr).predict_proba(X_va)[:, 1]
    print(f"[tab-only] {name:28s} n_feat {len(cols):2d}  AUC {roc_auc_score(y_va, p):.4f}")


In [24]:
# @@LOGIT_DECOMP@@ — exact logit decomposition: z = bias + image term + tabular term, no residual
# Loads one checkpoint per run and does a single forward pass over the adopted sample; the guard raises if the split does not reconstruct the prediction.

from isic24 import xai_logit as xlog

LOGIT_ALL_RUNS = False          # False = the six central settings (30 runs) | True = all 110

_lmaster = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
_central = None if LOGIT_ALL_RUNS else CENTRAL_SETTINGS
_by_run, _by_family, _per_sample = xlog.decompose_study(
    _lmaster, ART_PATH, subset_idx=xai_subset_idx, central=_central)

print("\n=== image share of the logit, by family ===")
print(_by_family.to_string(index=False))

# Three ways to ask "how much does the model use the image", side by side.
_m = _lmaster if _central is None else _lmaster[
    [bool(_central.get(i) == s) for i, s in zip(_lmaster.impl_name, _lmaster.setting_tag)]]
_cmp = (_by_run.groupby("impl_name")["mean_image_abs_share_descriptive"].mean().rename("logit_share")
        .to_frame()
        .join(_m.groupby("impl_name")[["img_drop_auc", "tab_drop_auc", "base_auc"]].mean()))
_cmp["ablation_share"] = _cmp.img_drop_auc / (_cmp.img_drop_auc + _cmp.tab_drop_auc)
print("\n=== three measures of reliance ===")
print(_cmp[["logit_share", "ablation_share", "img_drop_auc", "tab_drop_auc"]].round(3).to_string())
print(f"Spearman(logit_share, ablation_share) over {len(_cmp)} families: "
      f"{_cmp.logit_share.corr(_cmp.ablation_share, method='spearman'):+.3f}")

# Aggregating over both classes hides the asymmetry the strata already show.
print("\n=== per class: image share and target-aligned contribution ===")
print(_by_run.groupby("impl_name")[xlog.CLASS_COLS].mean().round(3).to_string())

# Consistency with the June vanilla decomposition (128 samples, same baselines).
_june = os.path.join(ART_PATH, "diagnostics", "vanilla_logit_contributions",
                     "20260604-210304", "vanilla_logit_contributions_by_seed.csv")
if os.path.exists(_june):
    _j = pd.read_csv(_june)["mean_image_abs_share_descriptive"].mean()
    _v = _by_run.loc[_by_run.impl_name == "vanilla", "mean_image_abs_share_descriptive"].mean()
    print(f"\n[check] vanilla image share — June n=128: {_j:.3f} | this sample n={len(xai_subset_idx)}: {_v:.3f}")


  [ablations] logit decomposition: 10 unimodal endpoint runs excluded from the aggregate
[1/30]              vanilla   baseline share 0.717  delta 1.2e-07
[2/30]              vanilla   baseline share 0.545  delta 1.2e-07
[3/30]              vanilla   baseline share 0.649  delta 1.5e-07
[4/30]              vanilla   baseline share 0.683  delta 6.0e-08
[5/30]              vanilla   baseline share 0.669  delta 1.2e-07
[6/30]        fixed_scaling     a0p500 share 0.756  delta 1.2e-07
[7/30]        fixed_scaling     a0p500 share 0.570  delta 1.2e-07
[8/30]        fixed_scaling     a0p500 share 0.686  delta 1.2e-07
[9/30]        fixed_scaling     a0p500 share 0.667  delta 1.2e-07
[10/30]        fixed_scaling     a0p500 share 0.729  delta 1.2e-07
[11/30]    learnable_scaling init_a0p500 share 0.656  delta 1.8e-07
[12/30]    learnable_scaling init_a0p500 share 0.650  delta 1.2e-07
[13/30]    learnable_scaling init_a0p500 share 0.673  delta 1.2e-07
[14/30]    learnable_scaling init_a0p500 share

In [ ]:
# @@GAMMA_ATTENDANCE@@ — attendance against the fusion bias, re-derived from the saved decomposition
_g_by_run, _g_by_family = xlog.gamma_curve(_per_sample, ART_PATH, master=_lmaster)

_neutral = _g_by_run[_g_by_run.delta == 0.0].set_index("run_id")["mean_image_abs_share_descriptive"]
_diff = (_by_run.set_index("run_id")["mean_image_abs_share_descriptive"] - _neutral).abs().max()
print(f"[check] delta=0 reproduces the measured decomposition: max |diff| = {_diff:.2e}")

print("\n=== image share of the logit against the fusion bias ===")
print(_g_by_family.pivot(index="impl_name", columns="gamma_tab",
                         values="mean_image_abs_share_descriptive__mean").round(3).to_string())

print("\n=== sample AUC against the fusion bias (adopted XAI sample, NOT the validation set) ===")
print(_g_by_family.pivot(index="impl_name", columns="gamma_tab",
                         values="sample_auc__mean").round(4).to_string())

_peak = _g_by_run.loc[_g_by_run.groupby(["impl_name", "seed"])["sample_auc"].idxmax()]
print("\n=== where each run's sample AUC peaks, by family ===")
print(_peak.groupby("impl_name")["delta"].agg(["mean", "std", "count"]).round(3).to_string())


## Report figures and the Grad-CAM overlap null

In [ ]:
# @@REPORT_FIGURES@@ — the sec. 3.6 figure, the one a table cannot carry (CPU, no checkpoint)
# Self-contained: paths and central settings come from the modules, so a fresh kernel is enough.
from isic24 import xai_figures as xfig

_arms = xfig.arms()
display(_arms.groupby("impl_name")[["del_exp", "del_rand", "del_gain",
                                    "ins_exp", "ins_rand", "ins_gain"]].mean().round(3))


In [ ]:
# @@CAM_OVERLAP@@ — do two families look in the same places more than two seeds do? (CPU)
# Reads the stored heatmaps only; the unit is the run pair, and the seed null is the comparison.
from isic24 import xai_metrics as xmet

_ov = xmet.cam_overlap_table(pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv")),
                             ART_PATH, central=xs.CENTRAL_SETTINGS)
display(_ov.groupby("pair_kind")[["iou_half", "iou_top"]].agg(["median", "mean", "count"]).round(3))


In [25]:
# @@SAMPLE_AUDIT@@ — do the four engines of a run explain the SAME rows? (CPU)
# Nesting is legitimate (LORE explains 48 of the 144); a partial overlap is two samples, one name.
_pop, _pop_ok = xs.sample_audit(pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv")),
                                ART_PATH)


              runs  n_min  n_max  knows_rows
kind                                        
faithfulness   110    128    128           0
gradcam_img    110    144    144         110
lore_sa        110     48     48         110
shap_tab       110    144    144         110
   110 runs  [SAMPLE] not one population — faithfulness: n=128 vs 144, rows unknown — it predates `val_index`, so nesting cannot be shown

[SAMPLE AUDIT] every population nested in the sample: FAIL


## Post-hoc diagnostics — gate faithfulness, calibration, gamma sweep

In [ ]:
# @@DIAG_HELPERS@@ — AUC, ablation and calibration helpers for the diagnostics cells

def eval_auc(model, ds):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(x, training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

# Build ablated inputs to remove/perturb one modality
def ablate_batch(x, mode, strength):
    x = dict(x)
    if mode in ("img", "both"):
        img = tf.cast(x["images"], tf.float32)
        x["images"] = (1.0 - strength) * img + strength * (tf.ones_like(img) * 0.5)  # gray image
    if mode in ("tab", "both"):
        feat = tf.cast(x["features"], tf.float32)
        mask = tf.random.uniform(tf.shape(feat)) >= strength
        x["features"] = tf.where(mask, feat, tf.zeros_like(feat))
    return x

# Evaluate AUC after modality ablation
def eval_auc_with_ablation(model, ds, mode, strength):
    m = keras.metrics.AUC()
    for x, y in ds:
        p = model(ablate_batch(x, mode, strength), training=False)
        m.update_state(y, p)
    return float(m.result().numpy())

# Simple Spearman (rank correlation) implementation
def spearman(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    ra = a.argsort().argsort().astype(np.float32)
    rb = b.argsort().argsort().astype(np.float32)
    ra -= ra.mean()
    rb -= rb.mean()
    return float((ra @ rb) / (np.linalg.norm(ra) * np.linalg.norm(rb) + 1e-12))

# Convert probabilities to logits for impact comparison
def logit(p, eps=1e-6):
    p = tf.clip_by_value(tf.cast(p, tf.float32), eps, 1 - eps)
    return tf.math.log(p / (1 - p))

# Gate-faithfulness: do gate weights correlate with true modality impact?
def gate_faithfulness(model, ds, strength=1.0):
    gate_model = keras.Model(model.inputs, model.get_layer("gate_weights").output)

    wimg, wtab, dimg, dtab = [], [], [], []
    for x, _ in ds:
        p0 = tf.squeeze(model(x, training=False), -1)
        l0 = logit(p0)

        # Impact after image ablation
        pi = tf.squeeze(model(ablate_batch(x, "img", strength), training=False), -1)
        di = tf.abs(l0 - logit(pi))

        # Impact after tabular ablation
        pt = tf.squeeze(model(ablate_batch(x, "tab", strength), training=False), -1)
        dt = tf.abs(l0 - logit(pt))

        # Predicted gate weights
        w = gate_model(x, training=False)
        wimg.append(w[:, 0].numpy())
        wtab.append(w[:, 1].numpy())
        dimg.append(di.numpy())
        dtab.append(dt.numpy())

    wimg = np.concatenate(wimg)
    wtab = np.concatenate(wtab)
    dimg = np.concatenate(dimg)
    dtab = np.concatenate(dtab)

    return {
        "spearman_w_img_delta_img": spearman(wimg, dimg),
        "spearman_w_tab_delta_tab": spearman(wtab, dtab),
    }

# Calibration metrics: Brier score + ECE
def brier_ece(model, ds, n_bins=15):
    y_all, p_all = [], []
    for x, y in ds:
        p = tf.squeeze(model(x, training=False), -1)
        y_all.append(tf.cast(tf.squeeze(y), tf.float32).numpy())
        p_all.append(tf.cast(p, tf.float32).numpy())

    y = np.concatenate(y_all)
    p = np.concatenate(p_all)
    brier = float(np.mean((p - y) ** 2))

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    N = len(p)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if np.any(m):
            conf = float(np.mean(p[m]))
            acc = float(np.mean(y[m]))
            ece += (np.sum(m) / N) * abs(acc - conf)

    return {"brier": brier, "ece": float(ece)}

In [ ]:
# @@GATE_DIAGNOSTICS@@ — gate weights, ablation and calibration per adaptive-gating run
all_diag = []
for _, row in ag_combined.iterrows():
    ag_run_dir = row["run_dir"]
    # combined_summary.csv stores June paths (/home/claudio/...); remap the
    # "/runs/..." suffix onto this machine's RUNS_PATH so checkpoints load.
    if not os.path.isdir(ag_run_dir):
        _s = str(ag_run_dir).replace("\\", "/")
        _i = _s.find("/runs/")
        if _i >= 0:
            ag_run_dir = os.path.join(os.path.dirname(RUNS_PATH), _s[_i + 1:])
    best_ckpt = os.path.join(ag_run_dir, "checkpoints", f"best_{row['model_name']}.keras")
    ag_model = keras.models.load_model(best_ckpt, compile=False, safe_mode=False)
    assert_fs_matches_model(ag_model, feature_space, row.get("run_id"))

    gate_model = keras.Model(ag_model.inputs, ag_model.get_layer("gate_weights").output)
    gx, _ = next(iter(validation_ds))
    gw = gate_model(gx, training=False).numpy()

    base_auc = eval_auc(ag_model, validation_ds)
    img_auc = eval_auc_with_ablation(ag_model, validation_ds, "img", 1.0)
    tab_auc = eval_auc_with_ablation(ag_model, validation_ds, "tab", 1.0)

    diag = {
        "run_id": row["run_id"],
        "setting_tag": row["setting_tag"],
        "seed": int(row["seed"]),
        "entropy_weight": float(row["entropy_weight"]),
        "run_dir": ag_run_dir,
        "gate_weight_mean_img": float(np.mean(gw[:, 0])),
        "gate_weight_mean_tab": float(np.mean(gw[:, 1])),
        "base_auc": base_auc,
        "img_drop_auc": base_auc - img_auc,
        "tab_drop_auc": base_auc - tab_auc,
        **gate_faithfulness(ag_model, validation_ds, strength=1.0),
        **brier_ece(ag_model, validation_ds),
    }

    os.makedirs(os.path.join(ag_run_dir, "diagnostics"), exist_ok=True)
    with open(os.path.join(ag_run_dir, "diagnostics", "adaptive_gating_diagnostics.json"), "w") as f:
        json.dump(diag, f, indent=2)
    all_diag.append(diag)

    # ---- diagnostic plots (per run), saved to the run's plots/ folder ----
    plots_dir = os.path.join(ag_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    # gather full-validation gate weights, labels and predictions in one pass
    wimg_all, wtab_all, y_true_all, p_pred_all = [], [], [], []
    for xb, yb in validation_ds:
        wb = gate_model(xb, training=False).numpy()
        pb = tf.squeeze(ag_model(xb, training=False), -1).numpy()
        wimg_all.append(wb[:, 0]); wtab_all.append(wb[:, 1])
        y_true_all.append(np.asarray(tf.squeeze(yb)).reshape(-1))
        p_pred_all.append(np.asarray(pb).reshape(-1))
    wimg_all = np.concatenate(wimg_all); wtab_all = np.concatenate(wtab_all)
    y_true_all = np.concatenate(y_true_all); p_pred_all = np.concatenate(p_pred_all)

    # 1) gate-weight distribution
    plt.figure(figsize=(7, 4))
    plt.hist(wimg_all, bins=30, alpha=0.6, label="w_img")
    plt.hist(wtab_all, bins=30, alpha=0.6, label="w_tab")
    plt.xlabel("gate weight"); plt.ylabel("count")
    plt.title(f"Adaptive gating: gate-weight distribution (seed {int(row['seed'])})")
    plt.legend(); plt.grid(alpha=0.3)
    _finalize_plot(plots_dir, "gate_weight_distribution.png")

    # 2) modality ablation: base AUC and AUC drop after ablating each modality
    plt.figure(figsize=(7, 4))
    labels = ["base_auc", "img_drop_auc", "tab_drop_auc"]
    vals = [diag["base_auc"], diag["img_drop_auc"], diag["tab_drop_auc"]]
    plt.bar(labels, vals, color=["#4c72b0", "#dd8452", "#55a868"])
    for j, v in enumerate(vals):
        plt.text(j, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    plt.ylabel("AUC"); plt.grid(alpha=0.3, axis="y")
    plt.title(f"Adaptive gating: modality ablation (seed {int(row['seed'])})")
    _finalize_plot(plots_dir, "modality_ablation_auc_drop.png")

    # 3) calibration reliability diagram
    n_bins = 15
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_conf, bin_obs = [], []
    for j in range(n_bins):
        lo, hi = edges[j], edges[j + 1]
        m = (p_pred_all >= lo) & (p_pred_all < hi) if j < n_bins - 1 else (p_pred_all >= lo) & (p_pred_all <= hi)
        if np.any(m):
            bin_conf.append(float(np.mean(p_pred_all[m])))
            bin_obs.append(float(np.mean(y_true_all[m])))
    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
    plt.plot(bin_conf, bin_obs, marker="o", label="model")
    plt.xlabel("mean predicted probability"); plt.ylabel("observed frequency")
    plt.title(f"Calibration (Brier={diag['brier']:.3f}, ECE={diag['ece']:.3f})")
    plt.legend(); plt.grid(alpha=0.3)
    _finalize_plot(plots_dir, "calibration_reliability.png")

    keras.backend.clear_session()
    gc.collect()

ag_diag_df = pd.DataFrame(all_diag)
ag_diag_df.to_csv(os.path.join(ag_study_dir, "adaptive_gating_diagnostics.csv"), index=False)
display(ag_diag_df)

In [ ]:
# @@CF_GAMMA_SWEEP@@ — post-hoc gamma sweep over every controlled-fusion run (GPU)
# gamma_tab > 1 enhances tabular and penalises image; gamma_tab < 1 does the opposite.
DELTAS = np.round(np.arange(-0.5, 0.501, 0.125), 3)
POSTHOC_SETTINGS = []
for d in DELTAS:
    POSTHOC_SETTINGS.append({
        "tag": f"d{d:+.3f}".replace(".", "p").replace("+", "plus").replace("-", "minus"),
        "delta": float(d),
        "gamma_img": float(1.0 - d),
        "gamma_tab": float(1.0 + d),
        "mode": "tab_enhance_img_penalize" if d > 0 else ("tab_penalize_img_enhance" if d < 0 else "neutral"),
    })

posthoc_rows = []
for _, row in cf_combined.iterrows():
    cf_run_dir = row["run_dir"]
    ckpt_path = os.path.join(cf_run_dir, "checkpoints", f"best_{row['model_name']}.keras")
    best_model = keras.models.load_model(ckpt_path, compile=False, safe_mode=False)
    assert_fs_matches_model(best_model, feature_space, row.get("run_id"))
    fusion_layer = best_model.get_layer("fusion_with_gamma")

    run_rows = []
    for s in POSTHOC_SETTINGS:
        fusion_layer.gamma_img.assign(s["gamma_img"])
        fusion_layer.gamma_tab.assign(s["gamma_tab"])
        auc_val = eval_auc(best_model, validation_ds)

        rec = {
            "run_id": row["run_id"],
            "setting_tag": row["setting_tag"],
            "seed": int(row["seed"]),
            "run_dir": cf_run_dir,
            "posthoc_id": s["tag"],
            "delta": s["delta"],
            "gamma_img": s["gamma_img"],
            "gamma_tab": s["gamma_tab"],
            "mode": s["mode"],
            "val_auc": auc_val,
            "base_model_name": row["model_name"],
        }
        run_rows.append(rec)
        posthoc_rows.append(rec)

    posthoc_df = pd.DataFrame(run_rows).sort_values("gamma_tab").reset_index(drop=True)
    diagnostics_dir = os.path.join(cf_run_dir, "diagnostics")
    os.makedirs(diagnostics_dir, exist_ok=True)
    posthoc_df.to_csv(os.path.join(diagnostics_dir, "posthoc_gamma_summary.csv"), index=False)

    plots_dir = os.path.join(cf_run_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    plt.figure(figsize=(7, 4))
    plt.plot(posthoc_df["gamma_tab"], posthoc_df["val_auc"], marker="o")
    plt.axvline(1.0, linestyle="--", linewidth=1)
    plt.xlabel("gamma_tab")
    plt.ylabel("val_auc")
    plt.title(f"Controlled fusion post-hoc: val_auc vs gamma_tab (seed {int(row['seed'])})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "posthoc_val_auc_vs_gamma_tab.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(7, 4))
    plt.scatter(posthoc_df["gamma_img"], posthoc_df["gamma_tab"], c=posthoc_df["val_auc"])
    for _, r in posthoc_df.iterrows():
        plt.text(r["gamma_img"], r["gamma_tab"], f"{r['val_auc']:.3f}", fontsize=8)
    plt.xlabel("gamma_img")
    plt.ylabel("gamma_tab")
    plt.title(f"Controlled fusion post-hoc sweep (seed {int(row['seed'])})")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, "posthoc_gamma_sweep_scatter.png"), dpi=300)
    plt.close()

    keras.backend.clear_session()
    gc.collect()

posthoc_all_df = pd.DataFrame(posthoc_rows).sort_values(["seed", "gamma_tab"]).reset_index(drop=True)
posthoc_all_df.to_csv(os.path.join(cf_study_dir, "controlled_fusion_posthoc_gamma_summary.csv"), index=False)

posthoc_agg = (
    posthoc_all_df.groupby(["delta", "gamma_img", "gamma_tab"], dropna=False)["val_auc"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
posthoc_agg.to_csv(os.path.join(cf_study_dir, "controlled_fusion_posthoc_gamma_aggregated.csv"), index=False)
display(posthoc_all_df.head(20))
display(posthoc_agg)

## Explanation faithfulness (deletion / insertion AUC)

`faithfulness_gain = deletion_auc(explanation) − deletion_auc(random)`, computed for `tabular` (SHAP), `image` (Grad-CAM) and `joint`. It reuses the SHAP arrays already on disk, so the GradientExplainer is not re-fitted. The three cells below define the loop, run it over every trained run and aggregate the result.


In [ ]:
# @@FAITH_LOOP@@ — post-hoc deletion/insertion faithfulness over all trained runs (GPU)
# Discovers run dirs from single_run_summary.csv; saves per-run artifacts and a combined CSV.
FAITH_RUN_GLOB = os.path.join(RUNS_PATH, "**", "single_run_summary.csv")  # any depth: 3-level single + 4-level multi-seed
FAITH_OUT_DIR = os.path.join(RUNS_PATH, "_faithfulness")
os.makedirs(FAITH_OUT_DIR, exist_ok=True)
_faith_name = "faithfulness" if globals().get("EVAL_IS_CANONICAL", True) else f"faithfulness_{EVAL_SET}"
_faith_methods = None if globals().get("EVAL_IS_CANONICAL", True) else ("image",)

_SETTING_PARAM_COLS = ("alpha", "init_alpha", "lambda_reg",
                       "gamma_img", "gamma_tab", "entropy_weight")

summ_paths = sorted(glob.glob(FAITH_RUN_GLOB, recursive=True))

selected_runs = []
n_reuse = 0
n_compute = 0
for sp in summ_paths:
    try:
        row = pd.read_csv(sp).iloc[0].to_dict()
    except Exception as e:
        print("  skip (bad csv):", sp, e)
        continue

    impl = row.get("impl_name")
    if FAITH_ONLY_IMPL and impl not in FAITH_ONLY_IMPL:
        continue

    run_dir = row.get("run_dir")
    if not isinstance(run_dir, str) or not os.path.isdir(run_dir):
        run_dir = os.path.dirname(sp)
    model_name = row.get("model_name")
    best_epoch = int(row.get("best_epoch", -1))
    ckpt = os.path.join(run_dir, "checkpoints", f"best_{model_name}.keras")
    if not os.path.exists(ckpt) or best_epoch < 0:
        print("  skip (no ckpt / best_epoch):", run_dir)
        continue

    fjson = os.path.join(run_dir, "xai", "faithfulness", f"{_faith_name}.json")
    will_reuse = os.path.exists(fjson) and not FAITH_OVERWRITE
    n_reuse += int(will_reuse)
    n_compute += int(not will_reuse)
    selected_runs.append((sp, row, run_dir, model_name, best_epoch, ckpt, fjson))

if FAITH_MAX_RUNS is not None:
    selected_runs = selected_runs[:int(FAITH_MAX_RUNS)]
    n_reuse = sum(os.path.exists(r[6]) and not FAITH_OVERWRITE for r in selected_runs)
    n_compute = len(selected_runs) - n_reuse

print(f"[FAITH] discovered {len(summ_paths)} run summaries | selected={len(selected_runs)} "
      f"| reuse={n_reuse} | compute={n_compute}")
if FAITH_DRY_RUN:
    display(pd.DataFrame([{"impl_name": r[1].get("impl_name"),
                           "setting_tag": r[1].get("setting_tag"),
                           "seed": r[1].get("seed"),
                           "run_dir": r[2],
                           "action": "reuse" if os.path.exists(r[6]) and not FAITH_OVERWRITE else "compute"}
                          for r in selected_runs]).head(50))
    raise SystemExit("FAITH_DRY_RUN=True: no models loaded.")

faith_rows = []
for sp, row, run_dir, model_name, best_epoch, ckpt, fjson in selected_runs:
    impl = row.get("impl_name")
    if os.path.exists(fjson) and not FAITH_OVERWRITE:
        res = json.load(open(fjson))
        res = {k: v for k, v in res.items() if k not in ("curves", "q_grid")}
    else:
        t0 = time.time()
        try:
            keras.backend.clear_session()
            model = keras.models.load_model(ckpt, compile=False, safe_mode=False)
            assert_fs_matches_model(model, feature_space, run_dir)
            res = explanation_faithfulness(model, run_dir, best_epoch, save=True, verbose=False,
                                           out_name=_faith_name, methods=_faith_methods)
        except Exception as e:
            print(f"  [WARN] {impl}/{row.get('setting_tag')}: {type(e).__name__}: {e}")
            keras.backend.clear_session(); gc.collect()
            continue
        keras.backend.clear_session(); gc.collect()
        res["faith_seconds"] = round(time.time() - t0, 2)

    rec = {
        "impl_name": impl,
        "setting_tag": row.get("setting_tag"),
        "seed": row.get("seed"),
        "best_val_auc": row.get("best_val_auc"),
        "run_dir": run_dir,
        **{c: row[c] for c in _SETTING_PARAM_COLS if c in row},
        **{k: v for k, v in res.items() if not isinstance(v, (list, dict))},
    }
    faith_rows.append(rec)
    print(f"  ok {impl:>18}/{str(row.get('setting_tag')):<14} "
          f"gain={res.get('faithfulness_gain', float('nan')):+.4f} "
          f"({res.get('faith_seconds', '?')}s)")

faith_df = pd.DataFrame(faith_rows)
out_csv = os.path.join(FAITH_OUT_DIR, f"combined_{_faith_name}.csv")
# Merge, not overwrite: keep prior rows for runs this pass did not recompute.
if os.path.exists(out_csv) and len(faith_df):
    _prev = pd.read_csv(out_csv)
    _prev = _prev[~_prev["run_dir"].isin(faith_df["run_dir"])]
    faith_df = pd.concat([_prev, faith_df], ignore_index=True)
elif os.path.exists(out_csv) and not len(faith_df):
    faith_df = pd.read_csv(out_csv)
faith_df.to_csv(out_csv, index=False)
print(f"[FAITH] saved {out_csv}  shape={faith_df.shape}")
display(faith_df.head(20))


In [ ]:
# @@MASTER_TABLE@@ — the run registry -> <artifacts>/summaries/
# One row per run plus a (family, setting) mean/std/count summary; needs combined_faithfulness.csv.
os.makedirs(SUMMARIES_PATH, exist_ok=True)
_frames = []
for _impl, _d in CANONICAL.items():
    _p = os.path.join(_d, "combined_summary.csv")
    if os.path.exists(_p):
        _df = pd.read_csv(_p); _df["impl_name"] = _impl; _frames.append(_df)
master = pd.concat(_frames, ignore_index=True)
_faith_csv = os.path.join(RUNS_PATH, "_faithfulness", "combined_faithfulness.csv")
if os.path.exists(_faith_csv):
    _fdf = pd.read_csv(_faith_csv)
    _keep = [c for c in ["run_dir","faithfulness_gain","faithfulness_tabular_del_gain",
             "faithfulness_image_del_gain"] if c in _fdf.columns]
    _fdf = _fdf[_keep].copy()
    # run_dir prefixes differ across machines: join on the suffix starting at "/runs/".
    def _relkey(p):
        p = str(p).replace("\\", "/")
        i = p.find("/runs/")
        return p[i:] if i >= 0 else p
    master["_key"] = master["run_dir"].map(_relkey)
    _fdf["_key"] = _fdf["run_dir"].map(_relkey)
    _fdf = _fdf.drop(columns=["run_dir"]).drop_duplicates("_key")
    _dupe = [c for c in _fdf.columns if c != "_key" and c in master.columns]
    master = master.drop(columns=_dupe).merge(_fdf, on="_key", how="left").drop(columns=["_key"])

# Fill lore_fidelity for the families whose runner did not log it inline.
import glob as _g
def _lkey(p):
    p = str(p).replace("\\", "/")
    i = p.find("/runs/")
    return p[i:] if i >= 0 else p
_lore_map = {}
_lp = os.path.join(RUNS_PATH, "_faithfulness", "lore_fidelity_posthoc.csv")
if os.path.exists(_lp):
    for _, _r in pd.read_csv(_lp).iterrows():
        _lore_map[_lkey(_r["run_dir"])] = float(_r["lore_fidelity"])
for _pe in _g.glob(os.path.join(CANONICAL["loss_regularization"], "**", "lore_fidelity_per_epoch.csv"), recursive=True):
    _e = pd.read_csv(_pe)
    if len(_e):
        _lore_map[_lkey(os.path.dirname(os.path.dirname(os.path.dirname(_pe))))] = float(_e.iloc[-1]["lore_fidelity_context"])
if _lore_map and "lore_fidelity" in master.columns:
    master["lore_fidelity"] = master["lore_fidelity"].fillna(master["run_dir"].map(_lkey).map(_lore_map))
_metrics = [c for c in ["best_val_auc","brier","ece","modality_balance","lore_fidelity",
            "faithfulness_gain","faithfulness_tabular_del_gain","faithfulness_image_del_gain"]
            if c in master.columns]
_group = [c for c in ["impl_name","setting_tag"] if c in master.columns]
summary = master.groupby(_group, dropna=False)[_metrics].agg(["mean","std","count"])
master.to_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"), index=False)
summary.to_csv(os.path.join(SUMMARIES_PATH, "master_results_summary.csv"))
print("[SUMMARY] wrote master_results_rows.csv + master_results_summary.csv ->", SUMMARIES_PATH)
display(summary)


In [ ]:
# @@STRATA_SECTION@@ 1/3 — build the full faithfulness table from the saved curves (CPU)
# Needs only the path and PARAMETERS cells; writes two derived CSVs, no model is loaded.
from isic24 import xai_faithfulness as xf

FAITH_STRATA_SIGNED = True    # include the signed variants (110 runs, 5 seeds per family)

_files = xf.discover_curve_files(ART_PATH, include_signed=FAITH_STRATA_SIGNED)
print("[curve] coverage by source:")
display(_files.groupby("source").agg(run=("run_id", "nunique"), file=("npz_path", "size")))

# Guard: the mean of the per-sample gains must reproduce the aggregate saved in the JSONs.
_probs = xf.verify_reintegration(_files[_files.source == "canonical"].npz_path.iloc[0])
print("[guard] verify_reintegration:", "OK (<1e-9)" if not _probs else f"** FAILED ** {_probs}")

strata = xf.stratified_table(ART_PATH, files=_files, q_maxes=tuple(Q_MAX_REPORT), verbose=False)
pooled = xf.pool_by_family(strata)

_p1 = os.path.join(RUNS_PATH, "_faithfulness", "faithfulness_strata.csv")
_p2 = os.path.join(RUNS_PATH, "_faithfulness", "faithfulness_strata_pooled.csv")
strata.to_csv(_p1, index=False)
pooled.to_csv(_p2, index=False)
print(f"\n[wrote] {_p1}   ({len(strata)} rows: one per run x source x cell)")
print(f"[wrote] {_p2}   ({len(pooled)} rows, pooled by family)")
print("\nBoth are derived: this cell regenerates them at any time, they are not canonical.")

In [ ]:
# @@STRATA_SECTION@@ 2/3 — summary table: one row per family, every attributor side by side
# Deletion, stratum "all", column gain_norm (per unit of budget); positive = faithful.
_h = pooled[(pooled.stratum == "all") & (pooled.direction == "del")].copy()
_h["attributore"] = np.where(
    _h.source == "canonical", "SHAPgrad:" + _h.method,
    np.where(_h.source == "aligned", "perm:tabular", "signed:" + _h["mode"]))

_piv = _h.pivot_table(index="impl_name", columns=["attributore", "q_max"], values="gain_norm")
print("gain_norm — deletion, all samples (positive = faithful)")
display(_piv.round(3))

# How many runs sit behind each number.
print("\nruns per cell (n_runs):")
display(_h.pivot_table(index="impl_name", columns="attributore", values="n_runs", aggfunc="max"))

print("attributors: SHAPgrad:* = GradientExplainer on the named branch | "
      "perm:tabular = logit-aligned Permutation | signed:* = the same, evidence for/against.")

In [ ]:
# @@STRATA_SECTION@@ 3/3 — strata: by class, and by confidence x correctness
# Shown at the low cutoff, the clean regime; at full budget the metric is OOD-dominated.
STRATA_SHOW = ["all", "class_pos", "class_neg",
               "sure_correct", "sure_wrong", "unsure_correct", "unsure_wrong"]
_QLOW = min(Q_MAX_REPORT)

_s = pooled[(pooled.direction == "del") & (pooled.q_max == _QLOW)
            & (pooled.stratum.isin(STRATA_SHOW))]

for _src, _mode, _meth, _title in [
        ("aligned", "abs", "tabular", "aligned Permutation — tabular, |attribution|"),
        ("canonical", "abs", "joint", "SHAP GradientExplainer — joint"),
        ("aligned_signed", "target", "tabular", "signed Permutation — true class (`target`)")]:
    _t = _s[(_s.source == _src) & (_s["mode"] == _mode) & (_s.method == _meth)]
    if not len(_t):
        print(f"\n[{_title}] no rows — source not on disk.")
        continue
    print(f"\n{_title}   (deletion, q<={_QLOW}, gain_norm)")
    display(_t.pivot_table(index="impl_name", columns="stratum", values="gain_norm")
              .reindex(columns=[c for c in STRATA_SHOW if c in set(_t.stratum)]).round(3))

print("\nEmpty cells are withheld on purpose below MIN_CELL_N; column `n` in `strata` "
      "gives the real sample count.")

## LORE explanations — regenerate them with the `lore_sa` engine

The only GPU-bound part of the post-hoc; everything else runs on the saved curves. These cells need the **full** setup (data, `FeatureSpace`, `xf.configure`), because the driver loads checkpoints and perturbs real features.

Measured cost: ~0.67 min per sample, so ~21 min per run at n=32.


In [ ]:
# @@LORE_SECTION@@ — aggregate the per-sample LORE CSVs the run above wrote (CPU)
# Two artifact paths exist; the newer xai/lore_sa/ wins per run, the historical one only fills gaps.
from isic24.run_ids import run_id_from_artifact

_nuovi = sorted(glob.glob(os.path.join(RUNS_PATH, "**", "xai", "lore_sa", "epoch*",
                                       "lore_sa_persample_*.csv"), recursive=True))
_storici = sorted(glob.glob(os.path.join(RUNS_PATH, "**", "xai", "lore", "epoch*",
                                         "tabular_sa", "lore_sa_persample_*.csv"),
                            recursive=True))
_visti = {run_id_from_artifact(f) for f in _nuovi}
_solo_storici = [f for f in _storici if run_id_from_artifact(f) not in _visti]
_lore_files = _nuovi + _solo_storici
print(f"[LORE] {len(_lore_files)} runs: {len(_nuovi)} from xai/lore_sa/ (new), "
      f"{len(_solo_storici)} from xai/lore/.../tabular_sa/ (historical, n=32)"
      + (f" — {len(_storici) - len(_solo_storici)} historical files skipped, the run "
         f"already has the new one" if len(_storici) != len(_solo_storici) else ""))
if _solo_storici and _nuovi:
    print("       [warn] the two sources differ in n and prevalence: compare the ordering, "
          "not the levels, until all runs are redone.")

if not _lore_files:
    print("No `lore_sa` artifact yet: run the cell above with LORE_DRY_RUN=False.")
else:
    # Run identity comes from `isic24.run_ids`, never from re-parsing the path by hand.
    _master = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
    _meta = {str(r.run_id): (r.impl_name, r.setting_tag) for _, r in _master.iterrows()}

    _rows, _unmatched = [], []
    for f in _lore_files:
        _rid = run_id_from_artifact(f)
        if _rid not in _meta:
            _unmatched.append(f"{_rid}   ({f})")
            continue
        d = pd.read_csv(f)
        d["run_id"] = _rid
        d["impl_name"], d["setting_tag"] = _meta[_rid]
        _rows.append(d)
    if _unmatched:
        raise RuntimeError("LORE artifacts absent from the canonical registry:\n  "
                           + "\n  ".join(_unmatched))

    L = pd.concat(_rows, ignore_index=True)
    L["covered"] = L["neigh_n_classes"] >= 2

    fam = L.groupby(["impl_name", "setting_tag"]).agg(
        n_run=("run_id", "nunique"),
        n_campioni=("i", "size"),
        lore_coverage=("covered", "mean"),                       # the attendance signal
        neigh_pos_rate=("neigh_pos_rate", "mean"),
        neigh_unique_frac=("neigh_unique_frac", "mean"),
        rule_len=("rule_len", "mean"),
        fid_cv_bacc=("fid_cv_bacc", "mean"),
    )
    # Fidelity only means something where the neighbourhood has two classes.
    fam["n_covered"] = L.groupby(["impl_name", "setting_tag"])["covered"].sum()
    fam["fid_cv_bacc_covered"] = (L[L.covered]
                              .groupby(["impl_name", "setting_tag"])["fid_cv_bacc"].mean())
    print("\nLORE explanation quality by family "
          "(`lore_coverage` = how far the tabular branch drives the decision)")
    display(fam.round(3))


    # Class-wise on the explanations, with the RUN as the unit: all runs share one sample,
    # so pooling rows across runs would inflate the p-value.
    cw = (L.groupby(["impl_name", "setting_tag", "true_label"])
            .agg(n=("i", "size"), lore_coverage=("covered", "mean"),
                 neigh_pos_rate=("neigh_pos_rate", "mean"),
                 rule_len=("rule_len", "mean"), fid_cv_bacc=("fid_cv_bacc", "mean"))
            .unstack("true_label"))
    print("\nthe same quantities split by true class (0 = benign, 1 = malignant)")
    display(cw.round(3))
    cw.reset_index().to_csv(
        os.path.join(RUNS_PATH, "_faithfulness", "lore_sa_quality_by_family_class.csv"),
        index=False)

    print("\nthe between-class contrast, tested across runs:")
    _per = L.groupby(["run_id", "true_label"])["covered"].mean().unstack().dropna()
    if len(_per) >= 2:
        xs.print_across_runs(xs.across_runs(_per[0], _per[1], "benign", "malignant"))
    else:
        print("  at least 2 runs with both classes are needed.")

    _out = os.path.join(RUNS_PATH, "_faithfulness", "lore_sa_quality_by_family.csv")
    fam.reset_index().to_csv(_out, index=False)
    print("wrote", _out)

## Distinct-lesion XAI sample

`pos_sample=5.0` uses `replace=True`, so 364 unique positives become 1820 rows and the same lesion is explained several times: the per-run confidence intervals are optimistic. The cells below build a sample of distinct lesions, check that its fingerprint round-trips through the saved SHAP arrays, measure how far the numbers move, and recompute SHAP, Grad-CAM, aligned faithfulness and LORE on it.

The saved npz store `explained_indices` as **positions**, so reusing them under a different sample would pair the SHAP values of some rows with the features of other lesions — silently, and with plausible numbers. That is what the fingerprint guard in `_load_shap_values` prevents.


In [ ]:
# @@SUBSET@@ 1/4 — probe: how much does the number move between the two samples? (GPU, ~8 min)
# Same run and attributor on the historical prefix and on the 144 distinct balanced lesions.
from isic24 import xai_driver as xd
import gc, keras, time, tempfile, shutil

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
_r = _m[(_m.impl_name == "vanilla")].iloc[0]
_rd = xs.run_dir_abs(_r.run_dir, ART_PATH)
# The probe is diagnostic: its outputs go to a temp dir, deleted at the end.
_out = tempfile.mkdtemp(prefix="probe_subset_")
_REF, _NEW = "PREFIX", "DISTINCT LESIONS"
_N = len(xai_subset_idx)
_ARMS = [(_REF, "prefix", np.arange(_N)),      # explicit: do NOT inherit the adopted default
         (_NEW, "distinct", np.asarray(xai_subset_idx))]
print(f"probe on {_r.run_id}  |  modes = {xf.SIGNED_MODES}  |  n = {_N}")
print(f"  indices shared by the two arms: "
      f"{len(np.intersect1d(_ARMS[0][2], _ARMS[1][2]))}/{_N}\n")

_model, _ = xd.load_run_model(_rd)
_res = {}
for _tag, _pfx, _si in _ARMS:
    _t0 = time.time()
    # Returns (results, phi), not a dict: unpack it.
    _o, _ = xf.aligned_signed_faithfulness(
        _model, _rd, int(_r.best_epoch), subset_idx=_si,
        out_dir=_out, out_prefix=f"probe_{_pfx}", verbose=False,
        q_maxes=tuple(Q_MAX_REPORT))
    _res[_tag] = {k: v for k, v in _o.items() if "gain" in str(k) and isinstance(v, float)}
    print(f"  {_tag:17s} {time.time()-_t0:5.0f}s", flush=True)
keras.backend.clear_session(); gc.collect()

# Guard: if the two arms saw the same sample, every delta below is zero by construction.
_z = {t: np.load(os.path.join(_out, f"probe_{p}_target_curves.npz")) for t, p, _ in _ARMS}
_same = np.array_equal(_z[_REF]["preds_orig"], _z[_NEW]["preds_orig"])
print(f"\n  positive prevalence: {_REF} {_z[_REF]['labels'].mean():.3f}  |  "
      f"{_NEW} {_z[_NEW]['labels'].mean():.3f}")
if _same:
    raise RuntimeError(
        "\n  THE TWO ARMS RAN ON THE SAME SAMPLE (identical preds_orig).\n"
        "  Known cause: `subset_idx=None` inherits `xf.FAITH_SUBSET_IDX`, set by\n"
        "  `xs.adopt()`. The reference arm must pass np.arange(n) explicitly.\n"
        "  No delta below would be interpretable.")
print("  [guard] the two arms are different samples — ok")

_cmp = pd.DataFrame({_REF: _res[_REF], _NEW: _res[_NEW]}).dropna()
_cmp.index = [str(i).split("aligned_")[-1] for i in _cmp.index]
_cmp["delta"] = _cmp[_NEW] - _cmp[_REF]
_cmp["rel_%"] = 100 * _cmp["delta"] / _cmp[_REF].abs().where(_cmp[_REF].abs() > 1e-9)
# `abs` sits at ~0.01 and jitters by as much between identical runs: its rel_% is noise
# over noise, so it is marked and must not be read as a shift.
_cmp["moves"] = np.where(_cmp.index.str.startswith("abs"), "(abs: below the noise)",
                 np.where(_cmp["delta"].abs() > 0.10, "<- MOVES", ""))
print(f"\n  shift ({_NEW.lower()} - {_REF.lower()}):")
display(_cmp.round(3))
_big = _cmp[(_cmp["delta"].abs() > 0.10) & (~_cmp.index.str.startswith("abs"))]
print(f"\n  verdict: the signal-carrying modes {'MOVE' if len(_big) else 'do not move'} "
      f"({len(_big)} non-abs metrics past 0.10 in absolute value)")
shutil.rmtree(_out, ignore_errors=True)


In [ ]:
# @@SUBSET@@ 2/4 — full XAI redo (Grad-CAM, SHAP, LORE) on the distinct-lesion sample (GPU)
# Resumable per run: interrupt at will, re-running picks up where it stopped.
from isic24 import xai_driver as xd
import gc, keras, time, os, numpy as np, pandas as pd

JOBS_TO_RUN = ["lore"]             # ["gradcam", "shap", "lore"]
ONLY_IMPL   = None                 # e.g. ["vanilla"]; None = every family
ONLY_SEEDS  = None                 # e.g. [47]; None = every seed
PER_FAMILY  = None                 # e.g. 1; None = every run
LORE_N      = 48
DRY_RUN     = False
LORE_ON_CPU = True

xd.LORE_SA_DEVICE = "/CPU:0" if LORE_ON_CPU else None

# Guard: with no adopted sample the engines would write into the canonical folders.
assert xd.shap_subdir() == "shap_tab" and xd.gradcam_subdir() == "gradcam_img", (
    "no sample adopted in this kernel: run the @@SETUP_XAI@@ cell, which calls `xs.adopt(...)`.\n"
    "  Without it SHAP and Grad-CAM would overwrite the canonical prefix artifacts.")

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
# Seeds are numbers in the master: coerce, so ["47"] does not silently select nothing.
if ONLY_SEEDS is not None: ONLY_SEEDS = [int(_s) for _s in ONLY_SEEDS]
if ONLY_IMPL:  _m = _m[_m.impl_name.isin([str(_x) for _x in ONLY_IMPL])]
if ONLY_SEEDS: _m = _m[_m.seed.astype(int).isin(ONLY_SEEDS)]
if PER_FAMILY: _m = _m.groupby("impl_name", sort=False).head(int(PER_FAMILY))
# Guard: an empty selection is not a successful pass, it is a pass that never started.
_all = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
if not len(_m):
    raise ValueError(
        f"\n  The filters select no run.\n"
        f"    ONLY_IMPL  = {ONLY_IMPL}      allowed: {sorted(_all.impl_name.unique())}\n"
        f"    ONLY_SEEDS = {ONLY_SEEDS}      allowed: {sorted(_all.seed.unique())}\n"
        f"    PER_FAMILY = {PER_FAMILY}\n"
        f"  (ONLY_IMPL takes a FAMILY name, not an engine: engines are chosen with "
        f"JOBS_TO_RUN.)")
_m = _m.reset_index(drop=True)
_RD = {r.run_id: xs.run_dir_abs(r.run_dir, ART_PATH)
       for _, r in _m.iterrows()}
print(f"selected {len(_m)} runs: {sorted(_m.impl_name.unique())} | seeds {sorted(_m.seed.unique())}")

# The LORE sample: half per class, evenly spaced inside each class so it is not a prefix.
_y = validation_df.target.values[np.asarray(xai_subset_idx)]
_lore_idx = np.sort(np.concatenate([
    np.asarray(xai_subset_idx)[_y == c][
        np.linspace(0, int((_y == c).sum()) - 1, LORE_N // 2).round().astype(int)]
    for c in (0, 1)]))
assert len(_lore_idx) == LORE_N and validation_df.target.values[_lore_idx].sum() == LORE_N // 2

# The plan, printed before the GPU is taken.
print(f"XAI sample: {XAI_SUBSET_FP}")
print(f"  SHAP and Grad-CAM on {len(xai_subset_idx)} lesions | LORE on {len(_lore_idx)} "
      f"({LORE_N//2} pos / {LORE_N//2} neg)\n")
_plan, _tot = {}, 0.0
for _j in JOBS_TO_RUN:
    _cfg = xs.JOBS[_j]
    _n = len(_lore_idx) if _j == "lore" else len(xai_subset_idx)
    _exp = _lore_idx if _j == "lore" else np.asarray(xai_subset_idx)
    _todo, _done, _redo = xs.resume_verified(_RD, _cfg["kind"], XAI_SUBSET_FP, _exp,
                                             task=_cfg["task"])
    _work = _todo + _redo               # `_redo` = stamped, but with a different sample
    _sec = _cfg["sec_per_sample"]
    _h = (len(_work) * _n * _sec / 3600) if _sec else None
    _tot += _h or 0
    _plan[_j] = (_work, set(_redo), _cfg, _n)
    print(f"  {_j:8s} -> xai/{_cfg['kind']:12s} done {len(_done):3d} | to do {len(_todo):3d}"
          + (f" | TO REDO {len(_redo):3d} (other sample)" if _redo else "")
          + (f" | ~{_h:.1f} h ({_n*_sec/60:.1f} min/run)" if _h else " | cost NOT MEASURED"))
print(f"\n  TOTAL ESTIMATE: ~{_tot:.1f} h" + ("  (+ Grad-CAM, not measured)"
      if any(xs.JOBS[j]["sec_per_sample"] is None for j in JOBS_TO_RUN) else ""))
print("  Interrupt at any time: re-running resumes where it stopped.")

if DRY_RUN:
    print("\n*** DRY RUN — nothing executed. Set DRY_RUN=False to run. ***")
else:
    _t0, _orphans = time.time(), []
    for _j in JOBS_TO_RUN:
        _todo, _redo_set, _cfg, _n = _plan[_j]
        if not _todo:
            continue
        print(f"\n===== {_j.upper()} — {len(_todo)} runs =====", flush=True)
        if _j == "lore":
            xd.LORE_SA_KWARGS["max_explain_samples"] = len(_lore_idx)
            xd.LORE_SA_KWARGS["subset_idx"] = _lore_idx
            print(f"  (LORE on {xd.LORE_SA_DEVICE or 'the default device = GPU'})",
                  flush=True)
        for _k, _rid in enumerate(_todo):
            _rd, _row = _RD[_rid], _m[_m.run_id == _rid].iloc[0]
            _t1 = time.time()
            try:
                # allow_clobber only for runs to redo; elsewhere the write guard stays armed.
                _o = xd.run_posthoc_xai(_rd, engines=_cfg["engines"],
                                        best_epoch=int(_row.best_epoch),
                                        best_val_auc=float(_row.best_val_auc),
                                        overwrite=True, verbose=False,
                                        allow_clobber=(_rid in _redo_set))
                # run_posthoc_xai does NOT raise on engine failure: it returns "failed: ...",
                # so a failed run must never be stamped as done.
                _bad = {k: v for k, v in (_o or {}).items() if str(v) != "ok"}
                if _bad or not _o:
                    print(f"  [FAIL] {_rid}: {_bad or 'no engine ran'}", flush=True)
                    continue
                xs.mark_done(_rd, _cfg["kind"], XAI_SUBSET_FP, task=_cfg["task"],
                             payload={"run_id": _rid, "impl_name": _row.impl_name,
                                      "setting_tag": _row.setting_tag, "n": int(_n),
                                      "result": str(_o)})
                print(f"  [{_k+1}/{len(_todo)}]{' REDONE' if _rid in _redo_set else ''} "
                      f"{_rid}: {_o}  "
                      f"({(time.time()-_t1)/60:.1f} min | {(time.time()-_t0)/60:.0f} tot)",
                      flush=True)
            except RuntimeError as _e:
                if "WRITE GUARD" in str(_e):
                    # Artifacts from an attempt killed before stamping: report, never clobber.
                    _orphans.append((_j, _rid, os.path.join(_rd, "xai", _cfg["kind"])))
                    print(f"  [ORPHANS] {_rid}: unstamped {_j} artifacts", flush=True)
                else:
                    print(f"  [FAIL] {_rid}: {type(_e).__name__}: {_e}", flush=True)
            except Exception as _e:
                print(f"  [FAIL] {_rid}: {type(_e).__name__}: {_e}", flush=True)
            finally:
                keras.backend.clear_session(); gc.collect()
        if _j == "lore":
            xd.LORE_SA_KWARGS.pop("subset_idx", None)

    print(f"\n===== closing ({(time.time()-_t0)/60:.0f} min) =====")
    for _j in JOBS_TO_RUN:
        _cfg = xs.JOBS[_j]
        _exp = _lore_idx if _j == "lore" else np.asarray(xai_subset_idx)
        _todo, _done, _redo = xs.resume_verified(_RD, _cfg["kind"], XAI_SUBSET_FP, _exp,
                                                 task=_cfg["task"])
        print(f"  {_j:8s} ok {len(_done)}/{len(_m)} on the right sample"
              + (f" — {len(_todo)+len(_redo)} left, re-run this cell"
                 if (_todo or _redo) else "  [complete]"))
    if _orphans:
        print(f"\n[WARN] {len(_orphans)} runs have ORPHAN artifacts, written by an attempt "
              f"killed before stamping. The write guard refuses to overwrite them.\n"
              f"  To redo them, delete the folder and re-run:")
        for _j, _rid, _p in _orphans[:10]:
            print(f"    rm -rf {_p}    # {_j} {_rid}")


In [ ]:
# @@SUBSET@@ 3/4 — aligned signed tabular faithfulness on the distinct-lesion sample (GPU)
# ~3.6 min/run at n=144; writes into <run_dir>/xai/shap_tab/ and resumes per run.
from isic24 import xai_driver as xd
import gc, keras, time, os, pandas as pd

STRAT_ONLY_IMPL  = None         # None = every family
STRAT_ONLY_SEEDS = None         # e.g. [47]; None = every seed
STRAT_PER_FAMILY = None         # e.g. 1 for one run per family; None = every run
STRAT_DRY_RUN    = False        # False to run

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
if STRAT_ONLY_IMPL:  _m = _m[_m.impl_name.isin(STRAT_ONLY_IMPL)]
if STRAT_ONLY_SEEDS: _m = _m[_m.seed.isin(STRAT_ONLY_SEEDS)]
if STRAT_PER_FAMILY: _m = _m.groupby("impl_name", sort=False).head(int(STRAT_PER_FAMILY))
_m = _m.reset_index(drop=True)

# run_id -> run_dir, re-anchored on ART_PATH: the master carries the paths of another machine.
_RD = {r.run_id: xs.run_dir_abs(r.run_dir, ART_PATH)
       for _, r in _m.iterrows()}

# 1.5 s/sample, measured with the four signed modes; changing SIGNED_MODES changes it.
SEC_PER_SAMPLE = 1.5
_todo, _done = xs.resume_report(_RD, "shap_tab", XAI_SUBSET_FP)
_est = len(_todo) * len(xai_subset_idx) * SEC_PER_SAMPLE
print(f"[subset] {len(_m)} runs selected | already done {len(_done)} | to do {len(_todo)}")
print(f"[subset] {len(xf.SIGNED_MODES)} modes x {len(xai_subset_idx)} samples "
      f"-> estimate {_est/60:.0f} min ({_est/3600:.1f} h)  [{_est/60/max(len(_todo),1):.1f} min/run]")
if _done:
    print(f"        (resume active: the {len(_done)} already done with fingerprint {XAI_SUBSET_FP} are skipped)")
if STRAT_DRY_RUN:
    print("\n*** DRY RUN — nothing executed. Set STRAT_DRY_RUN=False to run. ***")
else:
    _t0 = time.time()
    for _i, _r in _m.iterrows():
        _rd = _RD[_r.run_id]
        if xs.is_done(_rd, "shap_tab", XAI_SUBSET_FP):
            continue
        try:
            _model, _ = xd.load_run_model(_rd)
            # Returns (results, phi): unpack it, or mark_done is never reached.
            _o, _ = xf.aligned_signed_faithfulness(
                _model, _rd, int(_r.best_epoch), subset_idx=xai_subset_idx,
                verbose=False, q_maxes=tuple(Q_MAX_REPORT))
            # The stamp is written only here, once the run is complete: that makes it resumable.
            xs.mark_done(_rd, "shap_tab", XAI_SUBSET_FP,
                         {"run_id": _r.run_id, "impl_name": _r.impl_name,
                          "setting_tag": _r.setting_tag, "n": len(xai_subset_idx),
                          "gains": {k: v for k, v in _o.items() if isinstance(v, float)}})
            print(f"[{_i+1}/{len(_m)}] {_r.run_id}: ok  ({(time.time()-_t0)/60:.0f} min total)", flush=True)
        except Exception as _e:
            print(f"[FAIL] {_r.run_id}: {type(_e).__name__}: {_e}", flush=True)
        finally:
            keras.backend.clear_session(); gc.collect()
    _todo, _done = xs.resume_report(_RD, "shap_tab", XAI_SUBSET_FP)
    print(f"\n[subset] done {len(_done)}/{len(_m)} in {(time.time()-_t0)/60:.0f} min"
          + (f" — {len(_todo)} left, re-run this cell to resume" if _todo else ""))

    # Cross-family report, no GPU. `central` is not optional: a family mean over all 45
    # fixed_scaling rows, endpoints included, means nothing.
    if not _todo:
        xs.report(pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv")),
                  ART_PATH, kinds=("shap_tab",), central=CENTRAL_SETTINGS,
                  encoded_feature_names=encoded_feature_names)


In [ ]:
# @@SUBSET@@ 4/4 — LORE on the distinct-lesion sample (GPU)
# Half per class, evenly spaced inside each class; resumes per run.
from isic24 import xai_driver as xd
import gc, keras, time, numpy as np

# These filters keep the last selection used: re-read them before launching.
STRAT_LORE_ONLY_IMPL  = None        # None = every family
STRAT_LORE_ONLY_SEEDS = None        # None = every seed
STRAT_LORE_ONLY_TAG   = list(CENTRAL_SETTINGS.values())
STRAT_LORE_PER_FAMILY = 1
STRAT_LORE_DRY_RUN    = False
STRAT_LORE_N          = 48

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))
if STRAT_LORE_ONLY_IMPL:  _m = _m[_m.impl_name.isin(STRAT_LORE_ONLY_IMPL)]
if STRAT_LORE_ONLY_SEEDS: _m = _m[_m.seed.isin(STRAT_LORE_ONLY_SEEDS)]
if STRAT_LORE_ONLY_TAG:   _m = _m[_m.setting_tag.isin(STRAT_LORE_ONLY_TAG)]
if STRAT_LORE_PER_FAMILY: _m = _m.groupby("impl_name", sort=False).head(int(STRAT_LORE_PER_FAMILY))
_m = _m.reset_index(drop=True)
_RD = {r.run_id: xs.run_dir_abs(r.run_dir, ART_PATH)
       for _, r in _m.iterrows()}

# Half per class, evenly spaced inside each class, so the subset is not itself a prefix.
_y = validation_df.target.values[np.asarray(xai_subset_idx)]
_half = STRAT_LORE_N // 2
_pick = []
for _c in (0, 1):
    _pool = np.asarray(xai_subset_idx)[_y == _c]
    if len(_pool) < _half:
        raise ValueError(f"class {_c}: {len(_pool)} available, {_half} needed")
    _pick.append(_pool[np.linspace(0, len(_pool) - 1, _half).round().astype(int)])
_lore_idx = np.sort(np.concatenate(_pick))
_lore_y = validation_df.target.values[_lore_idx]
assert len(_lore_idx) == STRAT_LORE_N and _lore_y.sum() == _half, "the LORE sample is not 50/50"
print(f"[subset-lore] sample: {len(_lore_idx)} lesions, "
      f"{int(_lore_y.sum())} pos / {int((1-_lore_y).sum())} neg  (balanced)")

xd.LORE_SA_KWARGS["max_explain_samples"] = len(_lore_idx)

_todo, _done = xs.resume_report(_RD, "lore_sa", XAI_SUBSET_FP)
print(f"[subset-lore] {len(_m)} runs | already done {len(_done)} | to do {len(_todo)}")
MIN_PER_SAMPLE_LORE = 0.67       # minutes per sample, measured
print(f"[subset-lore] {len(_lore_idx)} samples/run -> estimate "
      f"{len(_todo)*len(_lore_idx)*MIN_PER_SAMPLE_LORE/60:.1f} h "
      f"({len(_lore_idx)*MIN_PER_SAMPLE_LORE:.0f} min/run)")
# A missing family or a degenerate representative must be seen now, not three hours in.
print(f"[subset-lore] families covered: {_m.impl_name.nunique()}/6")
_deg = _m[_m.setting_tag.astype(str).str.contains("a0p000|lam0e\\+00|init_a0p100")]
if len(_deg):
    print(f"\n[warn] {len(_deg)} selected runs are DEGENERATE settings (tabular branch zeroed "
          f"or lambda=0): a valid control, not a family representative.")
    display(_deg[["impl_name", "setting_tag", "run_id"]])
display(_m[["impl_name", "setting_tag", "seed", "run_id"]])

if STRAT_LORE_DRY_RUN:
    print("\n*** DRY RUN — nothing executed. Set STRAT_LORE_DRY_RUN=False to run. ***")
else:
    _prev = xd.LORE_SA_KWARGS.get("subset_idx")
    xd.LORE_SA_KWARGS["subset_idx"] = _lore_idx
    _t0 = time.time()
    try:
        for _i, _r in _m.iterrows():
            _rd = _RD[_r.run_id]
            if xs.is_done(_rd, "lore_sa", XAI_SUBSET_FP):
                continue
            try:
                # overwrite=True only disarms the xai_status.json shortcut, which would lie
                # here; the write guard stays armed and still refuses to clobber xai/lore_sa/.
                _o = xd.run_posthoc_xai(_rd, engines=("lore",), best_epoch=int(_r.best_epoch),
                                        best_val_auc=float(_r.best_val_auc),
                                        overwrite=True, verbose=False)
                xs.mark_done(_rd, "lore_sa", XAI_SUBSET_FP,
                             {"run_id": _r.run_id, "impl_name": _r.impl_name,
                              "setting_tag": _r.setting_tag, "n": int(len(_lore_idx)),
                              "n_pos": int(_lore_y.sum()), "result": str(_o)})
                print(f"[{_i+1}/{len(_m)}] {_r.run_id}: {_o}  "
                      f"({(time.time()-_t0)/60:.0f} min total)", flush=True)
            except Exception as _e:
                print(f"[FAIL] {_r.run_id}: {type(_e).__name__}: {_e}", flush=True)
            finally:
                keras.backend.clear_session(); gc.collect()
    finally:
        xd.LORE_SA_KWARGS["subset_idx"] = _prev   # do not leave the knob armed for other cells
    _todo, _done = xs.resume_report(_RD, "lore_sa", XAI_SUBSET_FP)
    print(f"\n[subset-lore] done {len(_done)}/{len(_m)} in {(time.time()-_t0)/60:.0f} min"
          + (f" — {len(_todo)} left, re-run to resume" if _todo else ""))
    if not _todo:
        xs.report(pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv")),
                  ART_PATH, kinds=("lore_sa",), central=CENTRAL_SETTINGS,
                  encoded_feature_names=encoded_feature_names,
                  lore_idx={len(_lore_idx): _lore_idx})


## Archived appendix — Stage 0–3 checks

The Stage 0/1/2/2b/3 checks and the legacy post-hoc LORE loops are no longer in this notebook. They live in git history (`git log --diff-filter=D -- scripts/archive_stage0_3.py`); their artifacts are intact under `runs/_robustness/stage2_aligned_attr/` and `runs/_robustness/stage3_image/`.


In [ ]:
# @@SHAPE@@ — backfill the explanation-shape metrics on the existing artifacts (CPU, ~10 s)
# Temporary: the 110 ISIC-24 runs predate these metrics being computed inside the engines.
from isic24 import xai_metrics as xm
import numpy as np, pandas as pd, os

_m = pd.read_csv(os.path.join(SUMMARIES_PATH, "master_results_rows.csv"))

# Which validation_df rows LORE explained, keyed by sample size; new runs write `val_index`
# themselves, this map only covers the older artifacts.
LORE_IDX_BY_N = {
    48: np.asarray(xai_subset_idx)[:48],   # prefix of the 144-lesion sample
    32: np.arange(32),                     # historical artifacts: raw prefix
}

_out = xs.report(_m, ART_PATH, kinds=(), central=CENTRAL_SETTINGS,
                 encoded_feature_names=encoded_feature_names, lore_idx=LORE_IDX_BY_N)
_shape, _stab, _fam = _out["shape"], _out["stability"], _out["by_family"]
print(f"\n[shape] {len(_shape)} runs | {len(_stab)} pairs | "
      f"{int(_shape.branch_off.sum())} runs with the tabular branch off (metrics = NaN)")

# Stability is only readable against its seed null: the Jaccard between two seeds of the
# same setting, on the same samples.
print("\nStability@k, against its seed null")
display(_stab.pivot_table(index="impl_name", columns="pair_kind", values="stability_at_k",
                          aggfunc=["mean", "count"]).round(3))
print("  `adjacent_param` > `same_param_seed` means moving the modality weight changes the")
print("  top-k features LESS than the seed does. Families with a single setting have only the null.")

print("\nall metrics side by side")
display(_fam.round(3))
